In [ ]:
#Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

In [ ]:
application_train = pd.read_csv('/content/application_train.csv')

In [ ]:
final_df = application_train.copy()

In [ ]:
bureau = pd.read_csv('/content/bureau.csv')

In [ ]:

bureau.info()

In [ ]:
bureau.head()


In [ ]:

bureau.shape


In [ ]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count'
})

In [ ]:
print(bureau_agg.head())

In [ ]:
bureau_agg.rename(columns={
    'SK_ID_BUREAU': 'TOTAL_BUREAU_LOANS'
}, inplace=True)

In [ ]:
print(bureau_agg.head())

In [ ]:
bureau['ACTIVE_LOAN'] = (bureau['CREDIT_ACTIVE'] == 'Active').astype(int)

In [ ]:
active_loans = bureau.groupby('SK_ID_CURR')['ACTIVE_LOAN'].sum()

In [ ]:
bureau_agg['ACTIVE_LOANS_COUNT'] = active_loans

In [ ]:
bureau_agg.head()

In [ ]:
overdue = bureau.groupby('SK_ID_CURR')['CREDIT_DAY_OVERDUE'].sum()

In [ ]:
bureau_agg['TOTAL_OVERDUE_DAYS'] = overdue

In [ ]:
bureau_agg.head()

In [ ]:
avg_debt = bureau.groupby('SK_ID_CURR')['AMT_CREDIT_SUM_DEBT'].mean()

In [ ]:
bureau_agg['AVG_DEBT'] = avg_debt

In [ ]:
max_credit = bureau.groupby('SK_ID_CURR')['AMT_CREDIT_SUM'].max()

In [ ]:
bureau_agg['MAX_CREDIT_SUM'] = max_credit

In [ ]:
bureau_agg.head()

In [ ]:
bureau_agg = bureau_agg.reset_index()

In [ ]:
bureau_agg.head()

In [ ]:
bureau_balance = pd.read_csv('/content/bureau_balance.csv')

In [ ]:
bureau_balance.head()


In [ ]:

bureau_balance.info()

In [ ]:
bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg({
    'MONTHS_BALANCE': 'count'
}).rename(columns={
    'MONTHS_BALANCE': 'MONTH_COUNT'
}).reset_index()

In [ ]:
bureau_balance['STATUS_NUM'] = bureau_balance['STATUS'].replace({
    'C': 0,
    'X': 0
}).astype(str)

bureau_balance['STATUS_NUM'] = bureau_balance['STATUS_NUM'].apply(
    lambda x: int(x) if x.isdigit() else 0
)

In [ ]:
bb_agg2 = bureau_balance.groupby('SK_ID_BUREAU')['STATUS_NUM'].max().reset_index()

In [ ]:
bb_loan_level = bb_agg.merge(bb_agg2, on='SK_ID_BUREAU', how='left')

In [ ]:
bureau_full = bureau.merge(bb_loan_level, on='SK_ID_BUREAU', how='left')

In [ ]:
bureau_balance_customer = bureau_full.groupby('SK_ID_CURR').agg({
    'MONTH_COUNT': 'mean',
    'STATUS_NUM': 'max'
}).reset_index()

In [ ]:
final_bureau = bureau_agg.merge(
    bureau_balance_customer,
    on='SK_ID_CURR',
    how='left'
)

In [ ]:
bureau_agg.head()

In [ ]:
final_bureau.head()

In [ ]:
final_df = final_df.merge(final_bureau, on='SK_ID_CURR', how='left')

In [ ]:
final_df.head()

In [ ]:
prev = pd.read_csv('/content/previous_application.csv')

In [ ]:
print(prev.shape)

In [ ]:
prev_agg = prev.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'AMT_APPLICATION': 'mean',
    'AMT_CREDIT': 'mean',
    'AMT_DOWN_PAYMENT': 'mean',
    'RATE_DOWN_PAYMENT': 'mean'
}).reset_index()

In [ ]:
prev_agg.rename(columns={
    'SK_ID_PREV': 'TOTAL_PREV_LOANS',
    'AMT_APPLICATION': 'AVG_APPLICATION_AMT',
    'AMT_CREDIT': 'AVG_APPROVED_CREDIT',
    'AMT_DOWN_PAYMENT': 'AVG_DOWN_PAYMENT',
    'RATE_DOWN_PAYMENT': 'AVG_DOWN_PAYMENT_RATE'
}, inplace=True)

In [ ]:
final_df = final_df.merge(prev_agg, on='SK_ID_CURR', how='left')

In [ ]:
CASH= pd.read_csv('/content/POS_CASH_balance.csv')

In [ ]:
CASH_agg = CASH.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'MONTHS_BALANCE': 'mean',
    'CNT_INSTALMENT': 'mean',
    'CNT_INSTALMENT_FUTURE': 'mean',
    'SK_DPD': ['max', 'mean', 'sum'],
    'SK_DPD_DEF': ['max', 'mean', 'sum']
}).reset_index()

In [ ]:
CASH_agg.columns = [
    'SK_ID_CURR',
    'TOTAL_POS_LOANS',
    'AVG_MONTHS_BALANCE',
    'AVG_INSTALLMENTS',
    'AVG_INSTALLMENTS_FUTURE',
    'POS_DPD_MAX',
    'POS_DPD_MEAN',
    'POS_DPD_SUM',
    'POS_DPD_DEF_MAX',
    'POS_DPD_DEF_MEAN',
    'POS_DPD_DEF_SUM'
]

In [ ]:
final_df = final_df.merge(CASH_agg, on='SK_ID_CURR', how='left')

In [ ]:
final_df.head()

In [ ]:
inst = pd.read_csv('/content/installments_payments.csv')

In [ ]:
inst_agg = inst.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'AMT_INSTALMENT': 'mean',
    'AMT_PAYMENT': 'mean',
    'DAYS_ENTRY_PAYMENT': 'mean',
    'DAYS_INSTALMENT': 'mean'
}).reset_index()

In [ ]:
inst_agg.columns = [
    'SK_ID_CURR',
    'TOTAL_INSTALMENTS',
    'AVG_INSTALMENT_AMOUNT',
    'AVG_PAYMENT_AMOUNT',
    'AVG_PAYMENT_DAY',
    'AVG_INSTALMENT_DAY'
]

In [ ]:
final_df = final_df.merge(inst_agg, on='SK_ID_CURR', how='left')

In [ ]:
cc = pd.read_csv('/content/credit_card_balance.csv')

In [ ]:
cc_agg = cc.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'MONTHS_BALANCE': 'mean',
    'AMT_BALANCE': 'mean',
    'AMT_CREDIT_LIMIT_ACTUAL': 'mean',
    'AMT_DRAWINGS_CURRENT': 'mean',
    'AMT_PAYMENT_TOTAL_CURRENT': 'mean',
    'CNT_DRAWINGS_CURRENT': 'mean',
    'SK_DPD': ['max', 'mean', 'sum'],
    'SK_DPD_DEF': ['max', 'mean', 'sum']
}).reset_index()

In [ ]:
cc_agg.columns = [
    'SK_ID_CURR',
    'TOTAL_CC_LOANS',
    'AVG_CC_MONTHS_BALANCE',
    'AVG_BALANCE',
    'AVG_CREDIT_LIMIT',
    'AVG_DRAWINGS',
    'AVG_PAYMENTS',
    'AVG_DRAWINGS_COUNT',
    'CC_DPD_MAX',
    'CC_DPD_MEAN',
    'CC_DPD_SUM',
    'CC_DPD_DEF_MAX',
    'CC_DPD_DEF_MEAN',
    'CC_DPD_DEF_SUM'
]

In [ ]:
final_df = final_df.merge(cc_agg, on='SK_ID_CURR', how='left')

In [ ]:
final_df.iloc[:, -10:].head()

In [ ]:

final_df.info()

In [ ]:
final_df.shape


In [ ]:

final_df.head()


In [ ]:
final_df['TARGET'].value_counts(normalize=True) * 100

In [ ]:
missing = final_df.isnull().mean() * 100
missing.sort_values(ascending=False).head(30)

In [ ]:
final_df = final_df.copy()

for col in final_df.columns:
    if final_df[col].isnull().sum() > 0:
        final_df[col + '_MISS'] = final_df[col].isnull().astype(int)

In [ ]:

zero_keywords = ['TOTAL', 'ACTIVE', 'OVERDUE', 'AVG_DEBT', 'MAX_CREDIT',
                 'POS', 'INSTALMENT', 'AMT_REQ', 'OBS', 'DEF', 'CC_DPD',
                 'STATUS_NUM', 'MONTH_COUNT', 'AVG_BALANCE', 'AVG_PAYMENTS',
                 'AVG_DRAWINGS', 'AVG_CREDIT_LIMIT', 'AVG_APPROVED',
                 'AVG_APPLICATION', 'AVG_DOWN', 'FLAG_DOCUMENT']

zero_cols = [col for col in final_df.columns
             if any(kw in col for kw in zero_keywords)
             and final_df[col].isnull().any()]

final_df[zero_cols] = final_df[zero_cols].fillna(0)


remaining_nulls = final_df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

num_remaining = final_df[remaining_nulls.index].select_dtypes(include=['int64','float64']).columns
cat_remaining = final_df[remaining_nulls.index].select_dtypes(include=['object']).columns

for col in num_remaining:
    final_df[col] = final_df[col].fillna(final_df[col].median())

for col in cat_remaining:
    final_df[col] = final_df[col].fillna('RARE')

final_df['FONDKAPREMONT_MODE'] = final_df['FONDKAPREMONT_MODE'].fillna('MISSING')

# validation
remaining = final_df.isnull().sum()
remaining = remaining[remaining > 0]
print(remaining)

In [ ]:
const_cols = [col for col in final_df.columns if final_df[col].nunique() <= 1]
final_df.drop(columns=const_cols, inplace=True)

In [ ]:
num_cols = final_df.select_dtypes(include=['int64','float64']).columns
cat_cols = final_df.select_dtypes(include=['object']).columns

In [ ]:
final_df[cat_cols].nunique().sort_values(ascending=False)

In [ ]:
top_vals = final_df['ORGANIZATION_TYPE'].value_counts().nlargest(10).index

final_df['ORGANIZATION_TYPE'] = final_df['ORGANIZATION_TYPE'].apply(
    lambda x: x if x in top_vals else 'RARE'
)

In [ ]:
for col in cat_cols:
    freq = final_df[col].value_counts(normalize=True)
    rare_labels = freq[freq < 0.01].index
    final_df[col] = final_df[col].replace(rare_labels, 'RARE')

In [ ]:
final_df.duplicated().sum()

In [ ]:
final_df.describe().T

In [ ]:
print("Rows:", final_df.shape[0])
print("Cols:", final_df.shape[1])

In [ ]:
# Feature engineering — credit risk ratios
final_df = final_df.copy()

new_features = pd.DataFrame({
    'CREDIT_INCOME_RATIO': final_df['AMT_CREDIT'] / (final_df['AMT_INCOME_TOTAL'] + 1),
    'ANNUITY_INCOME_RATIO': final_df['AMT_ANNUITY'] / (final_df['AMT_INCOME_TOTAL'] + 1),
    'CREDIT_GOODS_RATIO': final_df['AMT_CREDIT'] / (final_df['AMT_GOODS_PRICE'] + 1),
    'EMPLOYED_TO_AGE_RATIO': final_df['DAYS_EMPLOYED'] / (final_df['DAYS_BIRTH'] + 1),
    'INCOME_PER_PERSON': final_df['AMT_INCOME_TOTAL'] / (final_df['CNT_FAM_MEMBERS'] + 1)
})

final_df = pd.concat([final_df, new_features], axis=1)

In [ ]:
final_df.to_csv("finalmodel_dataset.csv", index=False)

In [ ]:
import os
os.listdir()